# 04 — Modeling

Train regression and classification models on both feature tracks.

This notebook retrains a **small** set of models so it stays fast. For the full catalog (Linear / Ridge / Random Forest / XGBoost, both tracks) run:

```bash
python3 -m src.pipeline
```

and read `results/model_performance.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_dataset
from src.feature_engineer import engineer_features
from src.metrics import regression_report
from src.models import make_demand_labels, train_classifiers, train_regressors
from src.preprocessor import temporal_split
from src.config import TARGET

featured = engineer_features(load_dataset())
split = temporal_split(featured)
y_train, y_test = split.train[TARGET], split.test[TARGET]
print("cutoff", split.cutoff.date())


## Approach 1 — demand regression


In [ ]:
op = train_regressors(split.train, split.test, y_train, y_test, "operational")
vendor = train_regressors(split.train, split.test, y_train, y_test, "vendor")
pd.DataFrame([m.metrics for m in op + vendor])


Operational models should land near the "half of inventory" baseline (R² ≈ 0.33). Vendor-track models should match the given forecast (R² ≈ 0.99, WAPE ≈ 6%). That is Forecast Value Added: does ML beat the forecast you already have?


## Approach 2 — LOW / MEDIUM / HIGH classification


In [ ]:
y_tr_c, y_te_c, low_q, high_q = make_demand_labels(y_train, y_test)
print("LOW <", low_q, "HIGH >", high_q)
clfs = train_classifiers(split.train, split.test, y_tr_c, y_te_c, "vendor")
pd.DataFrame([m.metrics for m in clfs])
